- Aqui tentarei repetir a análise de outliers, mas de maneira mais local, usando o algoritmo adequado, o Local Outlier Factor:

In [1]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from sklearn.neighbors import LocalOutlierFactor  
from sklearn.preprocessing import StandardScaler

# importação dos dados brutos 
ANO_ATUAL = 2026
ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()
df_pstats = nfl.load_player_stats(ultimos_5_anos).to_pandas()

# separando a temporada regular (para achar as anomalias) e o SB (para verificar sucesso)
df_reg = df_jogos[df_jogos['game_type'] == 'REG'].copy()
df_sb = df_jogos[df_jogos['game_type'] == 'SB'].copy()

# agregando estatísticas por jogo e time
df_pstats['turnovers_cometidos'] = df_pstats['passing_interceptions'] + df_pstats['fumbles_lost_total']

# somando a produção por time dentro de cada jogo específico
df_game_team = df_pstats.groupby(['season', 'game_id', 'team']).agg(
    Pass_Yds=('passing_yards', 'sum'),
    Rush_Yds=('rushing_yards', 'sum'),
    Carries=('carries', 'sum'), # controle de relógio
    Turnovers=('turnovers_cometidos', 'sum'),
    Sacks_Suffered=('sacks_suffered', 'sum') # base do Pass Rush
).reset_index()

# cruzando mandante e visitante
df_games = df_reg[['season', 'game_id', 'home_team', 'away_team', 'home_score', 'away_score']].copy()

# trazendo status do Mandante
df_games = df_games.merge(df_game_team, left_on=['game_id', 'home_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'home_pass', 'Rush_Yds':'home_rush', 'Carries':'home_carries', 
                         'Turnovers':'home_to', 'Sacks_Suffered':'home_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

# trazendo status do Visitante
df_games = df_games.merge(df_game_team, left_on=['game_id', 'away_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'away_pass', 'Rush_Yds':'away_rush', 'Carries':'away_carries', 
                         'Turnovers':'away_to', 'Sacks_Suffered':'away_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

df_games.fillna(0, inplace=True)

# construindo o perfil do time
# visão do Mandante
df_home_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['home_team'],
    'PF': df_games['home_score'], 'PA': df_games['away_score'],
    'Off_Pass_Yds': df_games['home_pass'], 'Off_Rush_Yds': df_games['home_rush'],
    'Off_Carries': df_games['home_carries'], 'Off_Turnovers': df_games['home_to'],
    'Def_Sacks_Produced': df_games['away_sacks'], # sacks que a defesa APLICOU
    'Def_Turnovers_Forced': df_games['away_to']   # turnovers que a defesa ROUBOU
})

# visão do Visitante
df_away_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['away_team'],
    'PF': df_games['away_score'], 'PA': df_games['home_score'],
    'Off_Pass_Yds': df_games['away_pass'], 'Off_Rush_Yds': df_games['away_rush'],
    'Off_Carries': df_games['away_carries'], 'Off_Turnovers': df_games['away_to'],
    'Def_Sacks_Produced': df_games['home_sacks'],
    'Def_Turnovers_Forced': df_games['home_to']
})

# agrupando a temporada inteira
df_season = pd.concat([df_home_persp, df_away_persp]).groupby(['season', 'team']).sum().reset_index()

In [2]:
# verificando o sucesso pelo superbowl
def status_superbowl(row):
    sb_ano = df_sb[df_sb['season'] == row['season']]
    if sb_ano.empty: return '-' 
    home, away = sb_ano.iloc[0]['home_team'], sb_ano.iloc[0]['away_team']
    vencedor = home if sb_ano.iloc[0]['home_score'] > sb_ano.iloc[0]['away_score'] else away
    if row['team'] == vencedor: return '🏆 Campeão'
    elif row['team'] in [home, away]: return '🥈 Vice'
    else: return '❌ Não Chegou'

df_season['Status_SB'] = df_season.apply(status_superbowl, axis=1)

# LOCAL OUTLIER FACTOR (LOF)
# o modelo foca nos pilares que vencem jogos
features = ['PF', 'PA', 'Off_Pass_Yds', 'Off_Rush_Yds', 'Off_Carries', 
            'Off_Turnovers', 'Def_Sacks_Produced', 'Def_Turnovers_Forced']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_season[features])

# n_neighbors define quantos "vizinhos" o algoritmo vai olhar para decidir se um time é anômalo.
# contamination=0.10 mantém o corte de 10% de anomalias na liga.
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.10)
df_season['Anomalia'] = lof.fit_predict(X_scaled)

# no LOF, não existe 'decision_function'. A métrica de anomalia é o "negative_outlier_factor_"
# quanto mais negativo o valor, mais isolado o time está em relação aos seus vizinhos diretos.
df_season['Score_Anomalia'] = lof.negative_outlier_factor_

In [3]:
# exibindo os outliers

colunas_off = ['team', 'PF', 'Off_Pass_Yds', 'Off_Rush_Yds', 'Off_Carries', 'Off_Turnovers', 'Status_SB', 'Anomalia']
colunas_def = ['team', 'PA', 'Def_Sacks_Produced', 'Def_Turnovers_Forced', 'Status_SB', 'Anomalia']

anos_unicos = sorted(df_season['season'].unique())

for ano in anos_unicos:
    df_ano = df_season[df_season['season'] == ano]
    
    # pegando os 3 melhores ataques (Mais pontos) e 3 melhores defesas (Menos pontos) daquela temporada
    top_off_ano = df_ano.sort_values(by='PF', ascending=False).head(3)
    top_def_ano = df_ano.sort_values(by='PA', ascending=True).head(3)
    
    print(f"\n{'='*30} TEMPORADA {ano} {'='*30}")
    
    print("🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)")
    print(top_off_ano[colunas_off].to_string(index=False))
    
    print("\n🧱 TOP 3 DEFESAS (Ordenadas por Pontos Sofridos)")
    print(top_def_ano[colunas_def].to_string(index=False))


============================== TEMPORADA 2021 ==============================
🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)
team  PF  Off_Pass_Yds  Off_Rush_Yds  Off_Carries  Off_Turnovers    Status_SB  Anomalia
 DAL 530          4963          2119          473             20 ❌ Não Chegou         1
  TB 511          5383          1672          385             19 ❌ Não Chegou         1
 BUF 483          4450          2209          461             22 ❌ Não Chegou         1

🧱 TOP 3 DEFESAS (Ordenadas por Pontos Sofridos)
team  PA  Def_Sacks_Produced  Def_Turnovers_Forced    Status_SB  Anomalia
 BUF 289                  42                    30 ❌ Não Chegou         1
  NE 303                  36                    30 ❌ Não Chegou         1
 DEN 322                  36                    19 ❌ Não Chegou         1

============================== TEMPORADA 2022 ==============================
🔥 TOP 3 ATAQUES (Ordenados por Pontos Feitos)
team  PF  Off_Pass_Yds  Off_Rush_Yds  Off_Carries  Off_

- Temporada 2021: A ilusão do jogo aéreo "normalizada"
    - O que aconteceu: Diferente do Isolation Forest, o LOF não caracterizou o Tampa Bay Buccaneers como uma aberração (-1). Avaliando a vizinhança local, o LOF considerou TB um time "normal" (1), mesmo com as 5.383 jardas.
    - Insight: Isso mostra que, embora TB fosse um ponto fora da curva global na década (Iso), localmente existiam outros times tentando imitar esse foco aéreo. O resultado, no entanto, foi o mesmo: a regra do FP-Growth se mantém. Um ataque puramente aéreo sem controle de relógio terrestre não é suficiente nem para chegar ao Super Bowl, seja ele uma anomalia global ou não.

- Temporada 2022: O rolo compressor terrestre incontestável
    - O que aconteceu: O Philadelphia Eagles chegou ao Super Bowl (Vice) mantendo seu selo de Anomalia Ofensiva histórica (-1). Mesmo avaliando a densidade dos vizinhos mais próximos, as 544 corridas e 2.509 jardas terrestres deles foram inigualáveis.
    - Insight: Esse é o combo do fp-growth '[Controle_Relogio] -> [Rush_Forte]' agindo na prática com dupla validação matemática. Eles literalmente esmagaram o cronômetro de uma forma que ninguém mais na liga conseguiu imitar. Mas, o Kansas City Chiefs foi o campeão operando como o melhor ataque não-anômalo, equilibrando perfeitamente seu ataque aéreo ao terrestre sem precisar de bizarrices estatísticas.

- Temporada 2023: O Pass Rush dita as regras (e as frustrações)
    - O que aconteceu: O cenário do Kansas City Chiefs (Campeão) construindo seu título na defesa se manteve, não sendo anômalo (1). O SF, o vice, também se manteve como (1). A grande confirmação é Baltimore: manteve-se como a única Anomalia Defensiva local e global (-1), com 60 sacks e 31 posses roubadas.
    - Insight: Validando brutalmente a regra '[Pass_Rush_Elite]'. A defesa dita o ritmo dos playoffs, mas a nota triste do Baltimore ganha ainda mais peso: você pode ter uma defesa tão absurda que quebra os modelos do Iso e do LOF ao mesmo tempo, mas sem um ataque minimamente capaz de capitalizar (e não 'pipocar', como muitos acham do QB Lamar Jackson), o time cai.

- Temporada 2024: A Obra-Prima do modelo (Refinando o Caos)
    - O que aconteceu: Aqui brilha a diferença entre os algoritmos, O Isolation Forest havia dito que DET, BUF e BAL eram anomalias. O LOF olhou a densidade estatística e disse o oposto: no "bairro" dos ataques explosivos, Detroit e Buffalo são apenas times muito bons e normais (1). A única anomalia ofensiva real (-1) foi Baltimore, por conta do extremo de 554 corridas para 3.189 jardas. E o Campeão? O Philadelphia Eagles manteve a coroa de única Defesa Anômala (-1) do ano (41 sacks e 26 turnovers).
    - Insight: O LOF filtrou perfeitamente a liga. Ele percebeu que ter ataques inflados não é anômalo, é apenas uma característica de alguns times, nessa temporada específica. Mas uma defesa implacável como a do Eagles foi solitária. Esse é o triunfo definitivo da defesa, previu a regra de Lift alto do fp-growth: enquanto a única anomalia ofensiva (BAL) falhou, a única anomalia defensiva (PHI) levantou o troféu.

- Temporada 2025: A "Condição de Perfeição" intocável
    - O que aconteceu: O Seattle Seahawks vence o Super Bowl de 2025 (3º melhor ataque, melhor defesa), não sendo uma anomalia matemática (-1) em nenhum dos lados da bola. A novidade do LOF foi capturar o Denver Broncos como uma Anomalia Defensiva (-1) por seus inacreditáveis 68 sacks, um time que ficou muito perto do superbowl, perdendo no round anterior em um jogo muito difícil em que seu QB se machucou.
    - Insight: A regra inicial '[Defesa_Elite, Ataque_Elite] -> [Vitoria]' de Confiança 1.00 se torna inquestionável. Seattle é a encarnação do equilíbrio. A sinergia perfeita entre dominar as trincheiras defensivas de forma sólida e controlar a bola no ataque sem forçar anomalias garantiu o título de Seattle.

- Resumidamente, o LOF limpou o ruído do Isolation Forest (como as "falsas" anomalias de TB em 21 e DET/BUF em 24), mas ambos os modelos confirmaram a mesma verdade por caminhos matemáticos diferentes. Um Super Bowl não se vence quebrando recordes estatísticos esparsos. Vence-se equilibrando Pass Rush (Sacks) com Controle de Relógio (Carries) em alta densidade estatística.